# Notebook 13: tuning the one thing that works

The multitask network is our strongest contributed arm and it has been run at exactly one
setting since the day it was written: learning rate 3e-4, 25 epochs, a 512-wide head,
dropout 0.1, weight decay 1e-5. Those were not chosen, they were typed. The only time
anything in this project was tuned, the gradient-boosting baseline, it gained +0.0135 on
that arm for nothing but compute.

Corpus scaling is dead, delta learning is dead, and the fifteen foundation encoders are
still being surveyed. This is the remaining lever that is certain to be a fair test,
because it varies a model we already know produces a useful arm.

**Two stages, so the compute goes where it matters.** Twelve configurations are screened
on a single fold, which is a fifth of the cost. The best two then get the full five folds
plus holdout and test, in the usual format, so they drop straight into the blend.

Screening is about 5 minutes a configuration and the full runs about 38 minutes each, so
roughly 2 to 2.5 hours unattended. Checkpointed throughout.

The honest expectation is +0.005 to +0.01 at the blend level. It is not a route to top
10%. It is the last untested thing about the model that actually carries weight.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout or 'NO GPU')

In [ ]:
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/tiny_6387286b00.tgz'

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
OUTDIR='/content/results'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR='/content/drive/MyDrive/cyp_results'
except Exception as e:
    print('Drive unavailable:', type(e).__name__)
os.makedirs(OUTDIR, exist_ok=True)
print('writing to', OUTDIR)

In [ ]:
!pip -q install chemprop==2.3.1

In [ ]:
import gc, json, time, urllib.request, numpy as np, pandas as pd, torch, torch.nn as nn
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

CACHE=Path('/content/cache'); OUT=Path(OUTDIR)
ISO=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
meta=pd.read_csv(CACHE/'meta_train.csv.gz'); mete=pd.read_csv(CACHE/'meta_test.csv.gz')
hold=np.load(CACHE/'holdout_mask.npy'); G=np.load(CACHE/'scaffold_groups.npy',allow_pickle=True)
AUX=np.load(CACHE/'aux_train.npy')
SM_TR=meta['SMILES'].to_numpy(); SM_TE=mete['SMILES'].to_numpy()

CK='/content/chemprop_medium.pt'
if not Path(CK).exists():
    urllib.request.urlretrieve('https://raw.githubusercontent.com/jeremycheminf/'
        'openadmet_scripts/main/CYP_Challenge/checkpoints/chemprop_medium.pt', CK)
print('checkpoint', round(Path(CK).stat().st_size/1e6,1), 'MB')

URL=('https://pubchem.ncbi.nlm.nih.gov/assay/pcget.cgi?query=download'
     '&record_type=datatable&actvty=all&response_type=save&aid=1851')
if not Path('/content/aid1851.csv').exists():
    urllib.request.urlretrieve(URL, '/content/aid1851.csv')
raw=pd.read_csv('/content/aid1851.csv', low_memory=False, skiprows=[1])
raw['pac50']=-pd.to_numeric(raw['Fit_LogAC50'], errors='coerce')
PANEL={'p450-cyp1a2':'CYP1A2','p450-cyp2c9':'CYP2C9','p450-cyp2c19':'CYP2C19',
       'p450-cyp2d6':'CYP2D6','p450-cyp3a4':'CYP3A4'}
raw['panel']=raw['Panel Name'].map(PANEL)
raw=raw.dropna(subset=['panel','pac50','PUBCHEM_EXT_DATASOURCE_SMILES'])
print('qHTS measurements', len(raw))

In [ ]:
def canon(s):
    m=Chem.MolFromSmiles(s) if isinstance(s,str) else None
    return Chem.MolToSmiles(m) if m is not None else None

ours={c for c in map(canon, np.concatenate([SM_TR,SM_TE])) if c}
pub=raw.groupby(['PUBCHEM_EXT_DATASOURCE_SMILES','panel'],as_index=False).pac50.median()
pub['canon']=[canon(s) for s in pub.PUBCHEM_EXT_DATASOURCE_SMILES]
pub=pub.dropna(subset=['canon'])
pub=pub[~pub.canon.isin(ours)]
wide=pub.pivot_table(index='PUBCHEM_EXT_DATASOURCE_SMILES',columns='panel',values='pac50')
PAN=['CYP1A2','CYP2C9','CYP2C19','CYP2D6','CYP3A4']
wide=wide.reindex(columns=PAN)
print('public compounds after dropping overlaps', len(wide))

HEADS=[f'chal_{i}' for i in ISO]+[f'scr_{i}' for i in ISO]+[f'qh_{p}' for p in PAN]
n_ch=len(SM_TR); n_pub=len(wide)
SM_ALL=np.concatenate([SM_TR, wide.index.to_numpy()])
Y=np.full((n_ch+n_pub, len(HEADS)), np.nan, np.float32)
for j,i in enumerate(ISO):
    Y[:n_ch, j]=meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float)
    Y[:n_ch, 4+j]=AUX[:, j]
Y[n_ch:, 8:]=wide.to_numpy(float)
M=np.isfinite(Y)
# Normalisation from development rows only; using all rows would fit a statistic on
# held-out labels. Affine, so it cannot inflate a Spearman, but it should still be clean.
_norm = Y.copy(); _norm[np.flatnonzero(hold)] = np.nan
mu = np.nanmean(_norm, 0); sd = np.nanstd(_norm, 0)
Yz_nan=((Y-mu)/np.where(sd>0,sd,1.0)).astype(np.float32)  # NaN kept as the mask
print('table', Y.shape, '| labels per head:', dict(zip(HEADS, M.sum(0).tolist())))

In [ ]:
from chemprop import data, featurizers
from chemprop.models.model import MPNN

FZ = featurizers.SimpleMoleculeMolGraphFeaturizer()

# Targets live inside the datapoints, so chemprop's own loader batches them with the
# graphs and nothing has to be re-featurised per minibatch. NaN marks "not measured"
# and becomes the loss mask.
PTS_ALL = [data.MoleculeDatapoint.from_smi(s, Yz_nan[i])
           for i, s in enumerate(SM_ALL)]
PTS_TE = [data.MoleculeDatapoint.from_smi(s, np.zeros(len(HEADS), np.float32))
          for s in SM_TE]
DS_TE = data.MoleculeDataset(PTS_TE, FZ)

def loader(idx=None, bs=128, shuffle=False, test=False):
    pts = PTS_TE if test else ([PTS_ALL[i] for i in idx] if idx is not None else PTS_ALL)
    return data.build_dataloader(data.MoleculeDataset(pts, FZ), num_workers=0,
                                 batch_size=bs, shuffle=shuffle, drop_last=False)

def _fp(m, bmg, V_d, X_d):
    # BatchMolGraph.to() moves in place and returns None, unlike a torch tensor, so it
    # must be called as a statement rather than used as an expression.
    if hasattr(bmg, 'to'):
        bmg.to(DEVICE)
    V_d = V_d.to(DEVICE) if torch.is_tensor(V_d) else V_d
    X_d = X_d.to(DEVICE) if torch.is_tensor(X_d) else X_d
    return m.fingerprint(bmg, V_d, X_d)

class Multi(nn.Module):
    def __init__(self, ckpt, n_out, warm=True, width=512, drop=0.1):
        super().__init__()
        m = MPNN.load_from_file(ckpt)
        if not warm:
            for p in m.parameters():
                nn.init.xavier_uniform_(p) if p.dim() > 1 else nn.init.zeros_(p)
        self.enc = m
        with torch.no_grad():                      # probe on CPU, before .to(DEVICE)
            bmg, V_d, X_d, *_ = next(iter(loader([0, 1, 2, 3], bs=4)))
            d = m.fingerprint(bmg, V_d, X_d).shape[1]
        self.head = nn.Sequential(nn.Linear(d, width), nn.ReLU(), nn.Dropout(drop),
                                  nn.Linear(width, n_out))

    def forward(self, bmg, V_d, X_d):
        return self.head(_fp(self.enc, bmg, V_d, X_d))

def train(tr_idx, epochs=25, warm=True, lr=3e-4, bs=128, seed=0,
          width=512, drop=0.1, wd=1e-5):
    torch.manual_seed(seed)
    m = Multi(CK, len(HEADS), warm, width, drop).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=wd)
    ld = loader(tr_idx, bs=bs, shuffle=True)
    for ep in range(epochs):
        m.train()
        for batch in ld:
            bmg, V_d, X_d, Yb, *_ = batch
            t = Yb.to(DEVICE).float()
            msk = torch.isfinite(t)
            if msk.sum() == 0:
                continue
            opt.zero_grad()
            p = m(bmg, V_d, X_d)
            loss = (((p - torch.nan_to_num(t)) ** 2) * msk).sum() / msk.sum()
            loss.backward()
            opt.step()
    return m

@torch.no_grad()
def predict(m, idx=None, bs=256, test=False):
    m.eval()
    out = [m(*batch[:3]).cpu().numpy() for batch in loader(idx, bs=bs, test=test)]
    return np.vstack(out) * sd + mu

## Stage 1, screen twelve configurations on one fold

Fold 0 only. The baseline configuration is included so the screen can be read against a
known quantity rather than only against itself.

In [ ]:
import pickle
GRID=[dict(name='baseline',     lr=3e-4, epochs=25, width=512,  drop=0.1,  wd=1e-5),
      dict(name='lr1e-3',       lr=1e-3, epochs=25, width=512,  drop=0.1,  wd=1e-5),
      dict(name='lr1e-4',       lr=1e-4, epochs=25, width=512,  drop=0.1,  wd=1e-5),
      dict(name='ep40',         lr=3e-4, epochs=40, width=512,  drop=0.1,  wd=1e-5),
      dict(name='ep15',         lr=3e-4, epochs=15, width=512,  drop=0.1,  wd=1e-5),
      dict(name='wide1024',     lr=3e-4, epochs=25, width=1024, drop=0.1,  wd=1e-5),
      dict(name='narrow256',    lr=3e-4, epochs=25, width=256,  drop=0.1,  wd=1e-5),
      dict(name='drop03',       lr=3e-4, epochs=25, width=512,  drop=0.3,  wd=1e-5),
      dict(name='drop00',       lr=3e-4, epochs=25, width=512,  drop=0.0,  wd=1e-5),
      dict(name='wd1e-3',       lr=3e-4, epochs=25, width=512,  drop=0.1,  wd=1e-3),
      dict(name='lr1e-3_ep40',  lr=1e-3, epochs=40, width=512,  drop=0.1,  wd=1e-5),
      dict(name='lr5e-4_wide',  lr=5e-4, epochs=30, width=1024, drop=0.2,  wd=1e-5)]

y_all={i: meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float) for i in ISO}
obs=np.isfinite(np.column_stack([y_all[i] for i in ISO])).any(1)
dev_rows=np.flatnonzero(obs & ~hold); ho_rows=np.flatnonzero(obs & hold)
pub_idx=np.arange(n_ch, len(SM_ALL))
folds=list(GroupKFold(5).split(dev_rows, dev_rows, G[dev_rows]))
tr0, va0 = folds[0]

SCR = OUT/'nb13_screen.pkl'
screen = pickle.load(open(SCR,'rb')) if SCR.exists() else {}
if screen: print('resuming;', len(screen), 'configurations already screened')

t0=time.time()
for g in GRID:
    if g['name'] in screen: continue
    kw={k:v for k,v in g.items() if k!='name'}
    try:                       # one bad configuration must not end the sweep
        m=train(np.concatenate([dev_rows[tr0], pub_idx]), **kw)
        p=predict(m, dev_rows[va0])[:, :4]
        rs=[]
        for j,i in enumerate(ISO):
            y=y_all[i]; q=dev_rows[va0]; ok=np.isfinite(y[q])
            rs.append(float(spearmanr(y[q][ok], p[ok, j]).statistic))
        screen[g['name']]=dict(macro=float(np.mean(rs)), per=rs, **kw)
        pickle.dump(screen, open(SCR,'wb'))
        print(f"  {g['name']:14s} macro {np.mean(rs):.4f}  "
              f"[{' '.join(f'{v:.3f}' for v in rs)}]  ({time.time()-t0:.0f}s)", flush=True)
        del m
    except Exception as e:
        print(f"  {g['name']:14s} FAILED  {type(e).__name__}: {str(e)[:70]}", flush=True)
    gc.collect(); torch.cuda.empty_cache()

assert 'baseline' in screen, 'the baseline configuration failed; nothing to compare against'
tab=pd.DataFrame(screen).T
tab['macro']=pd.to_numeric(tab['macro'])      # dict-of-dicts gives object columns
tab=tab.sort_values('macro', ascending=False)
print('\n'+tab[['macro','lr','epochs','width','drop','wd']].to_string())
base=screen['baseline']['macro']
BEST=[k for k in tab.index if k!='baseline'][:2]
print(f"\nbaseline {base:.4f} | carrying forward {BEST}")
print('If nothing beats the baseline by more than 0.005 on this fold, tuning is a null.')

## Stage 2, full runs for the best two

In [ ]:
CKPT=OUT/'nb13_state.pkl'
acc, done = ({}, set())
if CKPT.exists():
    st=pickle.load(open(CKPT,'rb')); acc=st['acc']; done=st['done']
    print('resuming;', len(done), 'fits complete')
def save(): pickle.dump({'acc':acc,'done':done}, open(CKPT,'wb'))

t0=time.time()
for name in BEST:
    kw={k:v for k,v in screen[name].items() if k in ('lr','epochs','width','drop','wd')}
    for f,(tr,va) in enumerate(folds):
        k=f'{name}_fold_{f}'
        if k in done: continue
        m=train(np.concatenate([dev_rows[tr], pub_idx]), **kw)
        acc[k]=predict(m, dev_rows[va])[:, :4]; done.add(k); save()
        print(f'  {name} fold {f} ({time.time()-t0:.0f}s)', flush=True)
        del m; gc.collect(); torch.cuda.empty_cache()
    for k, idx, extra in [(f'{name}_hold', ho_rows, {}), (f'{name}_test', None, {'test':True})]:
        if k in done: continue
        rows = np.concatenate([dev_rows, pub_idx]) if k.endswith('hold') else \
               np.concatenate([dev_rows, ho_rows, pub_idx])
        m=train(rows, **kw)
        acc[k]=predict(m, idx, **extra)[:, :4]; done.add(k); save()
        print(f'  {name} {k.split("_")[-1]} model ({time.time()-t0:.0f}s)', flush=True)
        del m; gc.collect(); torch.cuda.empty_cache()

RESULTS=[]
for name in BEST:
    oof=np.full((n_ch,4), np.nan, np.float32)
    for f,(tr,va) in enumerate(folds): oof[dev_rows[va]]=acc[f'{name}_fold_{f}']
    hp, tp = acc[f'{name}_hold'], acc[f'{name}_test']
    for j,i in enumerate(ISO):
        y=y_all[i]
        d=dev_rows[np.isfinite(y[dev_rows])]; h=ho_rows[np.isfinite(y[ho_rows])]
        np.savez_compressed(OUT/f'tuned_{name}_{i}.npz', oof=oof[d,j], dev=d,
                            hold=hp[np.isin(ho_rows,h),j], ho=h, test=tp[:,j])
        RESULTS.append(dict(cfg=name, iso=i,
                            dev_spearman=float(spearmanr(y[d],oof[d,j]).statistic),
                            holdout_spearman=float(spearmanr(y[h],hp[np.isin(ho_rows,h),j]).statistic)))
    json.dump(RESULTS, open(OUT/'results_nb13.json','w'), indent=2)
r=pd.DataFrame(RESULTS)
print('\n'+r.pivot(index='cfg',columns='iso',values='dev_spearman').round(4).to_string())
print('\nthe untuned multitask arm: dev 0.5577 0.6935 0.4515 0.8022, macro 0.6262')
import shutil
from google.colab import files
shutil.make_archive('/content/nb13_results','zip',OUT)
files.download('/content/nb13_results.zip')